In [4]:

import kagglehub

# Download latest version
path = kagglehub.dataset_download("metawave/vehicle-price-prediction")

print("Path to dataset files:", path)

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 28.3M/28.3M [00:06<00:00, 4.36MB/s]

Extracting files...


Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\metawave\vehicle-price-prediction\versions\1


In [10]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "vehicle_price_prediction.csv"
# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "metawave/vehicle-price-prediction",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

# print("First 5 records:", df.head())

C:\Users\User\AppData\Local\Temp\ipykernel_28988\2297079989.py:9: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


In [13]:
# --- Data Split ---

# Training set: Rows 0 to 70,000 (inclusive)
train_df = df.iloc[0:70001] 

# Validation set: Rows 70,001 to 85,000 (inclusive)[cite: 1]
val_df = df.iloc[70001:85001]

# Test set: Rows 85,001 to the end (inclusive)[cite: 1]
test_df = df.iloc[85001:]

print("\nSplit Verification")
print(f"Training Set Shape: {train_df.shape}")
print(f"Validation Set Shape: {val_df.shape}")
print(f"Test Set Shape: {test_df.shape}")


Split Verification
Training Set Shape: (70001, 20)
Validation Set Shape: (15000, 20)
Test Set Shape: (914999, 20)


In [21]:
#Decision Tree Classification & Regression
import numpy as np
from collections import Counter

class Node:
    def __init__(self, feature_idx=None, feature_name=None, threshold=None, left=None, right=None, *, value=None):
        self.feature_idx = feature_idx
        self.feature_name = feature_name
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value # Prediction value if it's a leaf node

    def is_leaf_node(self):
        return self.value is not None

class DecisionTree:
    def __init__(self, task='classification', min_samples_leaf=5, max_depth=100):
        self.task = task
        self.min_samples_leaf = min_samples_leaf # Constraint 2
        self.max_depth = max_depth
        self.root = None

    def fit(self, X, y, feature_names=None):
        # Default feature names if none are provided
        if feature_names is None:
            feature_names = [f"feature_{i}" for i in range(X.shape[1])]
        self.feature_names = feature_names
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        # Stopping criteria
        if (depth >= self.max_depth or 
            n_labels == 1 or 
            n_samples < self.min_samples_leaf * 2): # Need enough to split into two leaves of size >= min_samples_leaf
            leaf_value = self._calculate_leaf_value(y)
            return Node(value=leaf_value)

        feat_idx, thresh = self._best_split(X, y)

        if feat_idx is None: # No valid split found that satisfies min_samples_leaf
            leaf_value = self._calculate_leaf_value(y)
            return Node(value=leaf_value)

        # Build child nodes
        left_idxs, right_idxs = self._split(X[:, feat_idx], thresh)
        left = self._build_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._build_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        
        return Node(feature_idx=feat_idx, feature_name=self.feature_names[feat_idx], threshold=thresh, left=left, right=right)

    def _best_split(self, X, y):
        best_gain = -float('inf')
        split_idx, split_threshold = None, None

        for feat_idx in range(X.shape[1]):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for thr in thresholds:
                left_idxs, right_idxs = self._split(X_column, thr)

                # --- CONSTRAINT 2: min_samples_leaf ---[cite: 1]
                # Discard any split that results in a leaf node with fewer than 5 samples
                if len(left_idxs) < self.min_samples_leaf or len(right_idxs) < self.min_samples_leaf:
                    continue

                if self.task == 'classification':
                    gain = self._gini_gain(y, y[left_idxs], y[right_idxs])
                else:
                    gain = self._ssr_reduction(y, y[left_idxs], y[right_idxs])

                # --- CONSTRAINT 1: Alphabetical Tie-Breaker ---[cite: 1]
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_threshold = thr
                elif gain == best_gain and split_idx is not None:
                    # If gains are identical, compare feature names alphabetically
                    current_name = self.feature_names[feat_idx]
                    best_name = self.feature_names[split_idx]
                    if current_name < best_name:
                        split_idx = feat_idx
                        split_threshold = thr

        return split_idx, split_threshold

    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _gini_impurity(self, y):
        _, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        return 1.0 - np.sum(probabilities ** 2)

    def _gini_gain(self, y, left_y, right_y):
        parent_gini = self._gini_impurity(y)
        n = len(y)
        n_l, n_r = len(left_y), len(right_y)
        if n_l == 0 or n_r == 0:
            return 0
        child_gini = (n_l / n) * self._gini_impurity(left_y) + (n_r / n) * self._gini_impurity(right_y)
        return parent_gini - child_gini

    def _ssr(self, y):
        if len(y) == 0: return 0
        mean_y = np.mean(y)
        return np.sum((y - mean_y) ** 2)

    def _ssr_reduction(self, y, left_y, right_y):
        parent_ssr = self._ssr(y)
        child_ssr = self._ssr(left_y) + self._ssr(right_y)
        return parent_ssr - child_ssr # We want to maximize the reduction in SSR

    def _calculate_leaf_value(self, y):
        if self.task == 'classification':
            counter = Counter(y)
            return counter.most_common(1)[0][0]
        else:
            return np.mean(y)

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
        if x[node.feature_idx] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [20]:
#Random Forest Wrapper
class RandomForest:
    def __init__(self, n_trees=10, task='classification', min_samples_leaf=5, max_depth=100):
        self.n_trees = n_trees
        self.task = task
        self.min_samples_leaf = min_samples_leaf
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y, feature_names=None):
        self.trees = []
        for _ in range(self.n_trees):
            tree = DecisionTree(
                task=self.task, 
                min_samples_leaf=self.min_samples_leaf, 
                max_depth=self.max_depth
            )
            # Bootstrap sampling: random rows with replacement
            n_samples = X.shape[0]
            idxs = np.random.choice(n_samples, n_samples, replace=True)
            
            tree.fit(X[idxs], y[idxs], feature_names)
            self.trees.append(tree)

    def predict(self, X):
        # Get predictions from all trees
        # shape of tree_preds will be (n_trees, n_test_samples)
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        
        # Swap axes so shape is (n_test_samples, n_trees)
        tree_preds = np.swapaxes(tree_preds, 0, 1)
        
        y_pred = []
        for tree_pred in tree_preds:
            if self.task == 'classification':
                # Majority vote for classification
                counter = Counter(tree_pred)
                y_pred.append(counter.most_common(1)[0][0])
            else:
                # --- RANDOM FOREST REGRESSOR ---[cite: 1]
                # Average of the trees' predictions
                y_pred.append(np.mean(tree_pred))
                
        return np.array(y_pred)

In [19]:
tree.fit(X_train, y_train_price)

NameError: name 'tree' is not defined

In [18]:
# Section C
# Extract numpy arrays from your Pandas DataFrames
# Make sure to use exact lowercase strings matching your dataset

X_train = train_df.drop(columns=['price', 'condition']).values
y_train_price = train_df['price'].values # For regression
features = train_df.drop(columns=['price', 'condition']).columns.tolist()

print("Features extracted successfully!")
print("X_train shape:", X_train.shape)
print("y_train_price shape:", y_train_price.shape)

Features extracted successfully!
X_train shape: (70001, 18)
y_train_price shape: (70001,)
